In [69]:
import pandas as pd
import pickle
import geopandas as gpd
from itertools import product
from datasets import load_dataset
import os

In [22]:
path_hex = "/mnt/raid1/MAAT/08.accessibility/Copenhagen/"
with open(path_hex + "zones_Copenhagen.pkl", 'rb') as f:
    hexes = pickle.load(f)

path_zones = "/mnt/raid1/MAAT/02.GMM/"
zones = gpd.read_file(path_zones + "GMM_TAZ.gdb")


path_access = "/mnt/raid1/MAAT/20.surrogate_data/cph/accessibility_withassignment/parquet"
ds = load_dataset("parquet", data_files=path_access + "/*.parquet")
ds.set_format("pandas")

TRANSPORT_MODES = ['CAR', 'BICYCLE', 'ON_FOOT']
POI_CATEGORIES  = ['cultural', 'education', 'green_space', 'health',
                   'public_spaces', 'public_transportation', 'sports']

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [70]:
# Canonical TAZ ordering used for every TAZ-level list in the output below.
# Restricted to the 277 TAZ zones that have hexes (same set the accessibility
# side needs) - one extra TAZ zone that has roads but zero hexes is dropped
# from water_depths too, so every column in a row shares one consistent order.
TAZ_IDS = sorted(hexes['taz_zoneid'].unique())

NETWORK_DIR = "/mnt/raid1/MAAT/network"
edges = {
    mode: pickle.load(open(f"{NETWORK_DIR}/{mode}_edges_Copenhagen.pkl", "rb"))
    for mode in TRANSPORT_MODES
}
edge_taz_ids = {mode: edges[mode]['region_id'].to_numpy() for mode in TRANSPORT_MODES}

cols = list(product(TRANSPORT_MODES, POI_CATEGORIES))
aligned_acc = hexes[['hex_id', 'taz_zoneid']].copy()

full_df = ds["train"].to_pandas()

sample_rows = []
for sample_id, row in full_df.iterrows():  # looping through all samples
    sample_row = {}

    # water depths: mean per TAZ, over the edges that lie in that TAZ
    for mode in TRANSPORT_MODES:
        wd_col = f'water_depths_{mode}'
        wd_taz = pd.Series(row[wd_col]).groupby(edge_taz_ids[mode]).mean().reindex(TAZ_IDS)
        sample_row[wd_col] = wd_taz.tolist()

    # accessibility: mean per TAZ, over the hexes that lie in that TAZ
    for mode, category in cols:
        col_name = f'cumulative_accessibility_{mode}_{category}'
        aligned_acc[col_name] = row[col_name]

    agg = aligned_acc.groupby('taz_zoneid', as_index=False).mean(numeric_only=True)
    agg = agg.set_index('taz_zoneid').reindex(TAZ_IDS)

    for mode, category in cols:
        col_name = f'cumulative_accessibility_{mode}_{category}'
        sample_row[col_name] = agg[col_name].tolist()

    sample_rows.append(sample_row)

taz_acc = pd.DataFrame(sample_rows)
taz_acc = taz_acc[list(ds["train"].column_names)]  # match the source parquet's column order exactly

taz_acc

,water_depths_CAR,cumulative_accessibility_CAR_green_space,cumulative_accessibility_CAR_public_transportation,cumulative_accessibility_CAR_cultural,cumulative_accessibility_CAR_sports,cumulative_accessibility_CAR_public_spaces,cumulative_accessibility_CAR_education,cumulative_accessibility_CAR_health,water_depths_BICYCLE,cumulative_accessibility_BICYCLE_green_space,...,cumulative_accessibility_BICYCLE_education,cumulative_accessibility_BICYCLE_health,water_depths_ON_FOOT,cumulative_accessibility_ON_FOOT_green_space,cumulative_accessibility_ON_FOOT_public_transportation,cumulative_accessibility_ON_FOOT_cultural,cumulative_accessibility_ON_FOOT_sports,cumulative_accessibility_ON_FOOT_public_spaces,cumulative_accessibility_ON_FOOT_education,cumulative_accessibility_ON_FOOT_health
0,"[0.02561586340578643, 0.025452249877837795, 0....","[473.8888888888889, 472.3529411764706, 476.0, ...","[354.55555555555554, 354.88235294117646, 352.6...","[1227.2222222222222, 1226.235294117647, 1231.5...","[501.77777777777777, 502.7647058823529, 500.66...","[280.55555555555554, 279.88235294117646, 280.8...","[790.1111111111111, 788.0588235294117, 793.5, ...","[199.22222222222223, 198.05882352941177, 199.8...","[0.0192458157780394, 0.012228179549598057, 0.0...","[159.11111111111111, 166.76470588235293, 178.8...",...,"[217.0, 228.35294117647058, 238.5, 244.2142857...","[77.55555555555556, 85.88235294117646, 90.0, 9...","[0.006037725712453897, 0.008687323585615822, 0...","[16.444444444444443, 19.823529411764707, 24.83...","[48.666666666666664, 55.411764705882355, 58.5,...","[33.111111111111114, 30.88235294117647, 33.333...","[8.11111111111111, 7.647058823529412, 7.666666...","[27.88888888888889, 34.411764705882355, 36.0, ...","[27.333333333333332, 31.823529411764707, 39.83...","[19.555555555555557, 21.11764705882353, 25.166..."
1,"[0.01872347741199424, 0.021123786039300783, 0....","[475.6666666666667, 473.4117647058824, 477.5, ...","[354.0, 354.0, 354.0, 354.0, 354.0, 354.0, 353...","[1225.4444444444443, 1219.3529411764705, 1233....","[503.1111111111111, 502.7647058823529, 506.0, ...","[279.22222222222223, 278.88235294117646, 279.5...","[792.7777777777778, 790.7058823529412, 796.666...","[200.0, 199.0, 200.83333333333334, 201.0, 200....","[0.015768502300210027, 0.009630497818349567, 0...","[165.77777777777777, 170.35294117647058, 183.3...",...,"[221.88888888888889, 232.64705882352942, 243.1...","[77.22222222222223, 85.76470588235294, 92.0, 8...","[0.0024013756811028437, 0.00622918052232088, 0...","[17.11111111111111, 21.11764705882353, 25.1666...","[49.55555555555556, 57.1764705882353, 58.66666...","[34.22222222222222, 32.470588235294116, 33.166...","[8.333333333333334, 7.882352941176471, 7.66666...","[28.444444444444443, 34.8235294117647, 36.1666...","[27.88888888888889, 32.470588235294116, 40.0, ...","[20.0, 21.235294117647058, 25.0, 23.9285714285..."
2,"[0.031476956923658254, 0.045381174271432734, 0...","[465.44444444444446, 465.47058823529414, 388.5...","[1208.5555555555557, 1209.1764705882354, 1009....","[276.0, 276.5882352941176, 230.33333333333334,...","[796.0, 795.8823529411765, 663.3333333333334, ...","[348.0, 348.0, 290.0, 348.0, 335.1481481481481...","[199.0, 199.0, 165.66666666666666, 199.0, 191....","[486.3333333333333, 487.05882352941177, 405.5,...","[0.04188440714799441, 0.030547803717652467, 0....","[114.22222222222223, 128.58823529411765, 141.3...",...,"[57.77777777777778, 66.6470588235294, 75.0, 76...","[82.77777777777777, 99.29411764705883, 106.833...","[0.04263464973907686, 0.037783029485051474, 0....","[10.11111111111111, 13.764705882352942, 18.5, ...","[19.333333333333332, 19.88235294117647, 24.666...","[19.11111111111111, 25.823529411764707, 28.333...","[15.555555555555555, 22.11764705882353, 26.5, ...","[28.333333333333332, 40.1764705882353, 41.1666...","[12.222222222222221, 18.058823529411764, 21.16...","[3.888888888888889, 5.0588235294117645, 5.5, 5..."
3,"[0.04858162231769824, 0.036900029460707896, 0....","[467.77777777777777

In [71]:
output_dir = "/mnt/raid1/MAAT/20.surrogate_data/cph/accessibility_withassignment/parquet_taz"
os.makedirs(output_dir, exist_ok=True)

taz_acc.to_parquet(os.path.join(output_dir, "Copenhagen_taz.parquet"))

# The order of every list above is implicit (matches the source convention) -
# save the TAZ id it corresponds to at each position, right next to the data.
with open(os.path.join(output_dir, "taz_ids.pkl"), "wb") as f:
    pickle.dump(TAZ_IDS, f)

output_dir

'/mnt/raid1/MAAT/20.surrogate_data/cph/accessibility_withassignment/parquet_taz'

In [96]:
output_dir = "/mnt/raid1/MAAT/20.surrogate_data/cph/accessibility_withassignment/parquet_taz_baseline"
os.makedirs(output_dir, exist_ok=True)

min_row = float('inf')
min_id = -1
for id, row in enumerate(taz_acc['water_depths_CAR']):
    row_sum = sum(row)
    if row_sum < min_row:
        min_row = row_sum
        min_id = id

taz_acc.iloc[[min_id]].to_parquet(os.path.join(output_dir, "Copenhagen_taz_pseudo_dry_baseline.parquet"))

